In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import os

In [2]:
data_path = r'C:\Users\Vivek\Desktop\college_project-main\data\german_credit.csv'

In [3]:
df = pd.read_csv(data_path)

In [4]:
df.shape

(1000, 10)

In [5]:
df.columns

Index(['Unnamed: 0', 'Age', 'Sex', 'Job', 'Housing', 'Saving accounts',
       'Checking account', 'Credit amount', 'Duration', 'Purpose'],
      dtype='object')

In [6]:
df.head()

,Unnamed: 0,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose
0,0,67,male,2,own,NaN,little,1169,6,radio/TV
1,1,22,female,2,own,little,moderate,5951,48,radio/TV
2,2,49,male,1,own,little,NaN,2096,12,education
3,3,45,male,2,free,little,little,7882,42,furniture/equipment
4,4,53,male,2,free,little,little,4870,24,car


# Dropping unneccessary column named Unnamed: 0

In [7]:
df.drop(columns=["Unnamed: 0"],inplace=True)

In [8]:
df.columns

Index(['Age', 'Sex', 'Job', 'Housing', 'Saving accounts', 'Checking account',
       'Credit amount', 'Duration', 'Purpose'],
      dtype='object')

In [9]:
df.head()

,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose
0,67,male,2,own,NaN,little,1169,6,radio/TV
1,22,female,2,own,little,moderate,5951,48,radio/TV
2,49,male,1,own,little,NaN,2096,12,education
3,45,male,2,free,little,little,7882,42,furniture/equipment
4,53,male,2,free,little,little,4870,24,car


# Creating synthetic Risk column..."

In [10]:
df["Risk"] = np.where(
    (df["Credit amount"] > df["Credit amount"].median()) &
    (df["Duration"] > df["Duration"].median()),
    1,   # High Risk
    0    # Low Risk
)

# "Risk distribution:"

In [11]:
df["Risk"].value_counts()

Risk
0    650
1    350
Name: count, dtype: int64

# Convert categorical variables

In [12]:

df["age"] = df["Age"]

df["sex"] = df["Sex"].apply(lambda x: 1 if x == "male" else 0)

df["job"] = df["Job"]

df["housing"] = df["Housing"].apply(lambda x: 1 if x == "own" else 0)

saving_map = {
    "little": 0,
    "moderate": 1,
    "rich": 2,
    None: 0
}

df["savingsLevel"] = df["Saving accounts"].map(saving_map).fillna(0)

checking_map = {
    "little": 0,
    "moderate": 1,
    "rich": 1,
    None: 0
}

df["checkingLevel"] = df["Checking account"].map(checking_map).fillna(0)

df["creditAmount"] = df["Credit amount"]


df["duration"] = df["Duration"]

## Seperate features and target for model training

In [13]:
FEATURE_COLUMNS = [
    "age",
    "sex",
    "job",
    "housing",
    "savingsLevel",
    "checkingLevel",
    "creditAmount",
    "duration"
]

X = df[FEATURE_COLUMNS]
y = df["Risk"]

In [16]:
X.head()

,age,sex,job,housing,savingsLevel,checkingLevel,creditAmount,duration,Risk
0,67,1,2,1,0.0,0.0,1169,6,0
1,22,0,2,1,0.0,1.0,5951,48,1
2,49,1,1,1,0.0,0.0,2096,12,0
3,45,1,2,0,0.0,0.0,7882,42,1
4,53,1,2,0,0.0,0.0,4870,24,1


In [15]:
X["Risk"] = y

C:\Users\Vivek\AppData\Local\Temp\ipykernel_3976\3153512017.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X["Risk"] = y


In [19]:
X.to_csv(r"C:\Users\Vivek\Desktop\college_project-main\data\check_model.csv")

# splitting data into training and testing set

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [22]:
print("training dataset size:")
X_train.shape

training dataset size:


(800, 8)

In [23]:
print("testing dataset size:")
X_test.shape

testing dataset size:


(200, 8)

# using ranodom forest as ml model

In [24]:
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

In [25]:
model.fit(X_train,y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

# evaultaing our model

In [26]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

In [27]:
y_pred

array([0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1,
       0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1,
       1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1,
       0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1,
       1, 1])

In [28]:
comparison_df = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_pred
})
comparison_df.reset_index(drop=True, inplace=True)

In [29]:
comparison_df

,Actual,Predicted
0,0,0
1,0,0
2,1,0
3,0,0
4,1,1
...,...,...
195,1,1
196,1,1
197,1,1
198,1,1


In [30]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

In [31]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision (macro):", precision_score(y_test, y_pred, average='macro'))
print("Recall (macro):", recall_score(y_test, y_pred, average='macro'))
print("F1-score (macro):", f1_score(y_test, y_pred, average='macro'))

Accuracy: 0.99
Precision (macro): 0.9927536231884058
Recall (macro): 0.984375
F1-score (macro): 0.9884138570269957


In [32]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Confusion Matrix:
[[136   0]
 [  2  62]]


In [33]:
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       136
           1       1.00      0.97      0.98        64

    accuracy                           0.99       200
   macro avg       0.99      0.98      0.99       200
weighted avg       0.99      0.99      0.99       200



In [34]:
print("\nROC AUC:", roc_auc_score(y_test, y_prob))



ROC AUC: 1.0


# saving our trained model as a file for later use

In [35]:
import pickle

In [36]:
model_name = "credit_risk_model.pkl"
path = r'C:\Users\Vivek\Desktop\college_project-main\model'
final_path = os.path.join(path,model_name)

In [37]:
print(final_path)

C:\Users\Vivek\Desktop\college_project-main\model\credit_risk_model.pkl


In [38]:
try:
    with open(final_path, "wb") as file:
        pickle.dump(model, file, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Data successfully saved to '{final_path}'")

except (OSError, pickle.PicklingError) as e:
    print(f"Error saving data: {e}")

Data successfully saved to 'C:\Users\Vivek\Desktop\college_project-main\model\credit_risk_model.pkl'
